In [ ]:
!pip install transformers datasets evaluate accelerate -q

In [ ]:
from datasets import load_dataset
from collections import Counter

dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")

train_data = dataset["train"]
test_data  = dataset["validation"]

labels = train_data["label"]
print(Counter(labels))
print(f"Training samples : {len(train_data)}")
print(f"Test samples     : {len(test_data)}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_test  = test_data.map(tokenize_function, batched=True)

tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./financial_sentiment_model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(f"Final Test Accuracy: {results['eval_accuracy']*100:.1f}%")
print(f"Loss: {results['eval_loss']:.4f}")

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

label_map = {
    "LABEL_0": "Negative",
    "LABEL_1": "Neutral",
    "LABEL_2": "Positive"
}

test_sentences = [
    "The company reported record-breaking profits this quarter.",
    "Stock prices fell sharply amid investor concerns.",
    "The board announced a routine management meeting.",
    "Inflation fears continue to weigh on markets.",
    "The startup secured $50 million in Series B funding."
]

for sentence in test_sentences:
    result = classifier(sentence)[0]
    label  = label_map[result["label"]]
    conf   = result["score"] * 100
    print(sentence)
    print(f"{label} ({conf:.1f}%)\n")

In [ ]:
model.save_pretrained("./my_financial_sentiment_model")
tokenizer.save_pretrained("./my_financial_sentiment_model")

In [ ]:
from huggingface_hub import login

login(token="")

HF_USERNAME = "addyrall"
MODEL_NAME = f"{HF_USERNAME}/financial-sentiment-distilbert"

model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")

model.push_to_hub(MODEL_NAME)
tokenizer.push_to_hub(MODEL_NAME)